<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Supervised Learning - XGBoost - Diamonds
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install
!uv pip install --system -q dtreeviz

In [ ]:
# Daten
from pandas import read_csv, DataFrame

# Preprocessing
from sklearn.preprocessing import OrdinalEncoder

# Modellvorbereitung
from sklearn.model_selection import train_test_split

# Modell (Regression)
from xgboost.sklearn import XGBRegressor

# Evaluation
from sklearn.metrics import r2_score, mean_absolute_error

# Visualisierung
import plotly.express as px
import plotly.subplots as sp

# Modell-Visualisierung / Diagnose
import dtreeviz
from yellowbrick.regressor import residuals_plot

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1 | Understand
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "price",
    ],
)

In [ ]:
data = df.copy()
target = data.pop("price")

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Pandas
</font></p>

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.groupby("cut").count()

In [ ]:
data.groupby("color").count()

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Plotly
</font></p>

In [ ]:
title_ = "Depth"
b1 = px.box(data["depth"], title=title_, width=600, height=600)

title_ = "Carat"
b2 = px.box(data["carat"], title=title_, width=600, height=600)

title_ = "Table"
b3 = px.box(data["table"], title=title_, width=600, height=600)

fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Depth", "Carat", "Table"))

for trace in b1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in b2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in b3.data:
    fig.add_trace(trace, row=1, col=3)

# Layout anpassen
fig.update_layout(width=1000, height=500, title_text="Box-Plots")

# Plot anzeigen
fig.show()

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

<p><font color='black' size="5">
Datentyp ermitteln
</font></p>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Train-Test-Set
</font></p>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42)

<p><font color='black' size="5">
Kodierung
</font></p>

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

encoder = OrdinalEncoder(categories=cat_seq)
encoder.fit(data_train[cat_col])
data_train[cat_col] = encoder.transform(data_train[cat_col])
data_test[cat_col] = encoder.transform(data_test[cat_col])

# 3 |  Modeling
---

 <p><font color='black' size="5">
Modellauswahl & Training
</font></p>



[Doku](https://xgboost.readthedocs.io/en/stable/index.html#)    
[Parameter](https://xgboost.readthedocs.io/en/latest/parameter.html)

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Valdiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🚀 Algorithmus: XGBoost (Gradient Boosting)</font></p>

XGBoost (eXtreme Gradient Boosting) kombiniert viele schwache Entscheidungsbäume zu einem starken Modell. Jeder neue Baum wird darauf trainiert, die Fehler (Residuen) des bisherigen Ensembles zu korrigieren. Das Ergebnis ist ein sehr leistungsstarkes Modell, das bei vielen Datenwettbewerben zu den Besten gehört.

**Analogie:** Wie ein Team, das nacheinander eine Aufgabe löst – jedes Mitglied korrigiert die Fehler des Vorgängers. Gemeinsam liegen sie deutlich richtiger.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `n_estimators` | Anzahl der Bäume |
| `learning_rate` | Schrittweite bei der Fehlerkorrektur |
| `max_depth` | Maximale Tiefe jedes Baums |
| `random_state` | Reproduzierbarkeit |

**In der Praxis relevant wenn:**
- Höchste Regressionsgenauigkeit auf tabellarischen Daten gefragt ist — XGBoost ist auf Datenwettbewerben das Standardmodell für strukturierte Daten
- Numerische und kategoriale Features (nach Kodierung) gemischt vorliegen — XGBoost ist sehr flexibel
- Nichtlineare Zusammenhänge vermutet werden, die einfachere Modelle nicht erfassen können

**Nicht geeignet wenn:**
- Das Modell vollständig interpretierbar sein muss → ein einzelner Entscheidungsbaum ist transparenter
- Sehr wenige Datenpunkte vorliegen (< 500) → Boosting-Methoden benötigen genug Daten für stabile Gradienten

**Typischer Fehler:**
`learning_rate` zu groß wählen (Standard: 0.3) — dann überspringt XGBoost das Optimum. Faustregel: kleinere `learning_rate` (z.B. 0.05) mit mehr `n_estimators` kombinieren und `early_stopping_rounds` nutzen.

In [ ]:
#@markdown   <p><font size="4" color='green'>  XGBoost – Boosting-Prinzip</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
graph LR
    subgraph Input [Eingabedaten]
        D[Features]
    end

    subgraph Ensemble [Boosting-Prozess: Sequenzielle Bäume]
        direction TB
        B1[Baum 1: Erste Korrektur]
        B2[Baum 2: Korrigiert Residuen]
        B3[Baum 3: Weitere Korrektur]
        BN[Baum N: Feinjustierung]

        B1 -->|Residuen / Gradienten| B2
        B2 -->|Residuen / Gradienten| B3
        B3 -->|...| BN
    end

    D --> B1
    D --> B2
    D --> B3
    D --> BN

    Base[Basiswert] --> Sum
    B1 --> Sum
    B2 --> Sum
    B3 --> Sum
    BN --> Sum

    Sum[Additive Modellsumme] --> Output((Kontinuierlicher<br/>Zahlenwert))

    style Ensemble fill:#f1f8e9,stroke:#33691e
    style Output fill:#fff3e0,stroke:#ef6c00,stroke-width:2px
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=1050))

In [ ]:
model = XGBRegressor(objective="reg:squarederror", random_state=42)

In [ ]:
model.fit(
    data_train, target_train,
    eval_set=[(data_train, target_train), (data_test, target_test)],
    verbose=False
)

# 4 | Evaluate
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>


<p><font color='black' size="5">
Prediction
</font></p>


In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Bestimmtheitsmass
</font></p>

In [ ]:
r2 = r2_score(target_train, target_train_pred)
print(f"-- Train --- Bestimmtheitsmass: {r2:5.2f}")

In [ ]:
r2 = r2_score(target_test, target_test_pred)
print(f"-- Test --- Bestimmtheitsmass: {r2:5.2f}")

<p><font color='black' size="5">
Mean Absolut Error
</font></p>

In [ ]:
mae = mean_absolute_error(target_test, target_test_pred)
print(f"-- Test -- Mean Absolute Error: {mae:5.2f}")

<p><font color='black' size="5">
Fortschritt über Boosting-Iterationen
</font></p>

In [ ]:
# Evaluierungsergebnisse abrufen
results = model.evals_result()

# DataFrame für Visualisierung erstellen
df_progress = DataFrame({
    "Iteration": range(len(results["validation_0"]["rmse"])),
    "Train": results["validation_0"]["rmse"],
    "Validate": results["validation_1"]["rmse"]
})

# Liniendiagramm mit plotly-express
fig = px.line(
    df_progress,
    x="Iteration",
    y=["Train", "Validate"],
    title="Fortschritt über Boosting-Iterationen (RMSE)",
    labels={"value": "RMSE", "variable": "Dataset"},
    width=800,
    height=500
)
fig.show()

<p><font color='black' size="5">
Aufbau Analysewürfel
</font></p>

In [ ]:
# Übernahme der Testdaten
cube = data_test.copy()
cube.reset_index(inplace=True)

# Übernahme Target real & predict
cube["real"]    = target_test.values
cube["predict"] = target_test_pred

<p><font color='black' size="5">
Visualisierung real vs predict
</font></p>

In [ ]:
# Boxplot
title_ = "Boxplot real vs predict"
px.box(cube[["real", "predict"]], title=title_, width=600, height=600)

In [ ]:
# Histogramm
title_ = "Histogramm Prices real vs predict"
fig = px.histogram(cube, x=["real", "predict"], nbins=10, text_auto=".2s", title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

<p><font color='black' size="5">
Fehlerhafte Vorhersagen
</font></p>

In [ ]:
cube["abs_Abw%"] = abs((cube["real"] - cube["predict"]) / cube["real"] * 100)
cube.head(10).style.format("{:,.1f}")

In [ ]:
cube.describe().T

In [ ]:
# Histogramm
title_ = "Histogramm absolute Abweichung"
fig = px.histogram(cube, x=["abs_Abw%"], nbins=20, text_auto=".2s", title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

<p><font color='black' size="5">
Feature Importance
</font></p>

In [ ]:
px.bar(
    x=model.feature_importances_,
    y=data.columns,
    text_auto=".2s",
    title="Feature Importance",
    width=800,
    height=500,
).update_yaxes(categoryorder="total ascending")

<p><font color='black' size="5">
Residuals Plot
</font></p>

In [ ]:
_ = residuals_plot(model, data_train, target_train, data_test, target_test)

<p><font color='black' size="5">
Entscheidungsbaum
</font></p>

In [ ]:
viz_model = dtreeviz.model(
    model, data_test, target_test, tree_index=1, feature_names=list(data.columns.values)
)

In [ ]:
viz_model.view(scale=1.0, fontname="Monospace", show_node_labels=True)

In [ ]:
viz_model.view(scale=1.0, orientation="LR", fontname="Monospace")

In [ ]:
# local Explanation
one = data_test.iloc[13]
viz_model.view(x=one, fontname="Monospace")

In [ ]:
# local Explanation
viz_model.view(x=one, show_just_path=True, fontname="Monospace")

In [ ]:
tree_img = viz_model.view(scale=0.8, fontname="Monospace")
tree_img.save("tree.svg")

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>